# Track 1 · 05 Train — ClipLoss probe, then LoRA

Two rungs in one file, because the second warm-starts from the first.

**Run `track1_02_features` first, in this session.** Section A needs
`features_pooled.npz` from Drive; Section B needs the EEG, which it re-epochs itself.

---

## Why this exists

The ridge head scored exactly chance (rank 0.4998, top-5 2.48% against a 2.50% floor),
and the scaling curve was flat from 3 to 12 subjects, so the problem is not data. Two
deviations from the reference remained:

**Wrong objective.** Ridge minimises `||pred - target||^2`. The metric is cosine
retrieval. Because DINOv2 embeddings share a large common component, the cheapest way to
reduce squared error is to predict something near the gallery mean, which is exactly what
we watched happen: a collapsed head scored 0.78 cosine while being useless. The reference
uses `ClipLoss` with `norm_kind: y`, `temperature: false`, `symmetric: false` — contrastive,
so it optimises *distinguishability* rather than absolute closeness.

**Wrong rung.** The reference recipe is "2-step (linear probe then LoRA on QKVO)". The
leaderboard agrees: an entry labelled `test reve 3 epochs` scores 0.1349, five times
chance, from three epochs of fine-tuning. Our frozen probe sits at chance.

Section A fixes the objective while everything stays frozen. Section B unfreezes the
attention projections. A is minutes on cached features; B is the expensive one.

## What actually trains

| | Section A | Section B |
|---|---|---|
| trainable | head, ~0.79M | head + LoRA, ~1.9M |
| frozen | all 69M encoder | all 69M encoder (LoRA adds, never overwrites) |
| input | cached 512-d features | raw windows, re-epoched |
| feature cache | used | useless, encoder changes every step |

LoRA freezes each `W` and learns `B·A` beside it with `B` initialised to zero, so training
starts exactly at the pretrained function and departs from there. Targets are
`transformer.layers.*.0.{to_qkv,to_out}`, the 44 attention projections named in
`reve.yaml`.

## One detail that would silently corrupt the loss

About 30% of windows repeat an image. Two windows of the *same* image landing in one batch
become negatives of each other, so the loss punishes the model for being right. Those pairs
are masked out.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

RUN_CLIP_PROBE = False
RUN_UNPOOLED = False    # Section D: pooling test, closed form (already run)
RUN_LORA = True         # Section B: the owed weight-decay run
MAKE_ZIP = False

SEED = 33
VAL_IMAGE_FRAC = 0.2

# Section A
A_EPOCHS, A_BATCH, A_LR = 50, 256, 1e-3

# Section B. One dict per run; the loop keeps whichever scores best on val rank.
# alpha is held at 2*rank so the LoRA scale (alpha/r) stays fixed and rank is the
# only thing varying.
B_BATCH = 32
CONFIG_SET = 'reg_sweep'        # 'reg_sweep' or 'baseline'

BASELINE = [dict(name='r16 lr1e-4 wd1e-2', rank=16, lr=1e-4, head_lr=1e-3,
                 wd=1e-2, epochs=8, norm_pred=False)]

# Every LoRA run so far used a mean-pooled head, and Section D then showed pooling
# is what costs us the signal:
#   REVE pooled 512    rank 0.4998      REVE unpooled 16,384  rank 0.5070
# So LoRA has never had access to the information. This is the untried combination:
# the best representation with the training method, rather than with closed-form
# ridge.
#
# Learned pooling rather than no pooling. A flatten head would be 25M parameters on
# 10,709 windows and is fixed to one channel count; a trained query is ~512
# parameters, weights electrodes instead of averaging them, and stays invariant to
# channel count so it still transfers to the warm-up corpus.
#
# One variable against the wd 1e0 run: pooling. Everything else identical.
REG_SWEEP = [
    dict(name='attention pooling', rank=8, lr=1e-4, head_lr=1e-3, wd=1e0,
         epochs=15, norm_pred=False, pooling='attention'),
]

NORM_PRED = False      # reve.yaml norm_kind: y; each config overrides this
ENCODER = 'brain-bzh/reve-base'
SUBMISSION_NAME = 'track1_reve_lora'
ZIP_DIR = '/content/submission'

In [ ]:
%pip install -q mne 'braindecode>=1.8' huggingface_hub safetensors

import importlib
for mod in ('mne', 'braindecode', 'torch'):
    m = importlib.import_module(mod)
    print(f'{mod:12} {getattr(m, "__version__", "ok")}')

In [ ]:
import os, json, time, shutil, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / 'Track1'
PREP, RAW, WEIGHTS = BASE / 'prepared', BASE / 'raw', BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
DATASET = 'nm000134'


def report(title, checks):
    print(title)
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    print(f'  -> {sum(c[1] for c in checks)} ok, {sum(not c[1] for c in checks)} failed')


def l2np(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)


def rank_pct(pred, y_ids, emb):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2np(pred) @ l2np(emb[gal]).T
    tc = sims[np.arange(len(sims)), truth]
    return float((sims < tc[:, None]).mean())


def top5(pred, y_ids, emb):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2np(pred) @ l2np(emb[gal]).T
    k = min(5, len(gal))
    topk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in r for t, r in zip(truth, topk)]), 100.0 * k / len(gal)


def score(pred, y_ids, subj, emb, label):
    """Per-window and within-subject-aggregated. Warm-up scores per-window."""
    acc, ch = top5(pred, y_ids, emb)
    r = rank_pct(pred, y_ids, emb)
    agg = {}
    for p_, y_, s_ in zip(pred, y_ids, subj):
        agg.setdefault((s_, y_), []).append(p_)
    ks = sorted(agg)
    pa = np.stack([np.mean(agg[k], 0) for k in ks])
    ya = np.array([k[1] for k in ks])
    acc_a, _ = top5(pa, ya, emb)
    print(f'  {label:<16} rank {r:.4f}   top-5 {acc:5.2f}% (per-window, = the warm-up '
          f'metric)   {acc_a:5.2f}% (subject-agg)   chance {ch:.2f}%')
    return dict(rank=r, top5=acc, top5_agg=acc_a, chance=ch)


RESULTS = WEIGHTS / 'results.json'


def save_result(name, res):
    all_ = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}
    all_[name] = res
    RESULTS.write_text(json.dumps(all_, indent=2))
    return all_


def load_results():
    return json.loads(RESULTS.read_text()) if RESULTS.exists() else {}


print(f'device {DEVICE}')
if DEVICE == 'cpu':
    print('No GPU. Section A is fine on CPU; Section B will be painfully slow.')

## Load the cached features and the target table

In [ ]:
f = np.load(PREP / 'features_pooled.npz', allow_pickle=True)
t = np.load(PREP / 'image_embeddings.npz', allow_pickle=True)
emb = l2np(t['emb'].astype('float32'))          # ClipLoss norm_kind: y
DIM = emb.shape[1]

Xtr, ytr, str_ = f['X_train'].astype('float32'), f['y_train'], f['subj_train']
Xte, yte, ste = f['X_test'].astype('float32'), f['y_test'], f['subj_test']

# Preprocessing settings travel with the artefact, so Section B cannot drift from
# whatever track1_02_features actually did.
PP = dict(filt=[float(x) for x in np.atleast_1d(f['filt'])] if f['filt'].size else None,
          baseline=str(f['baseline']), clamp=float(f['clamp']),
          scale_scope=str(f['scale_scope']), sfreq=float(f['sfreq']))
print('features', Xtr.shape, Xte.shape, ' targets', emb.shape)
print('preprocessing recorded in the bundle:', PP)
if PP['filt'] is None:
    print('\nWARNING: these features were built with no band-pass. Re-run')
    print('track1_02_features before trusting anything below.')

# image-wise validation split, so lambda/early stopping never sees a training image
imgs = np.unique(ytr)
val_imgs = set(rng.choice(imgs, int(len(imgs) * VAL_IMAGE_FRAC), replace=False).tolist())
is_val = np.array([y in val_imgs for y in ytr])
print(f'{len(imgs):,} train images -> {len(val_imgs):,} held out '
      f'({(~is_val).sum():,} fit / {is_val.sum():,} val windows)')

## Section A — ClipLoss on the frozen head

Same inputs and same `512 -> 1536` linear as the ridge fit. Only the objective changes.

In [ ]:
EMB_T = torch.tensor(emb, device=DEVICE)


def clip_loss(pred, target_idx):
    """Contrastive, brain-to-target only, fixed temperature, target-normalised.

    Mirrors reve.yaml: norm_kind y, temperature false, symmetric false.
    """
    Y = EMB_T[target_idx]                       # already unit norm
    P = F.normalize(pred, dim=1) if NORM_PRED else pred
    logits = P @ Y.T
    same = target_idx[:, None] == target_idx[None, :]
    same.fill_diagonal_(False)                  # repeats of one image are NOT negatives
    logits = logits.masked_fill(same, -1e4)
    return F.cross_entropy(logits, torch.arange(len(pred), device=pred.device))


def train_head(Xf, yf, Xv, yv, epochs, batch, lr, head=None, log_every=10):
    Xf_t = torch.tensor(Xf, device=DEVICE)
    yf_t = torch.tensor(yf.astype('int64'), device=DEVICE)
    Xv_t = torch.tensor(Xv, device=DEVICE)
    yv_t = torch.tensor(yv.astype('int64'), device=DEVICE)
    head = head or nn.Linear(Xf.shape[1], DIM).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-2)
    best, best_state = -1, None
    for ep in range(1, epochs + 1):
        head.train()
        perm = torch.randperm(len(Xf_t), device=DEVICE)
        tot = 0.0
        for i in range(0, len(perm) - 1, batch):
            idx = perm[i:i + batch]
            if len(idx) < 8:
                continue
            opt.zero_grad()
            loss = clip_loss(head(Xf_t[idx]), yf_t[idx])
            loss.backward(); opt.step()
            tot += float(loss) * len(idx)
        head.eval()
        with torch.no_grad():
            out_v = head(Xv_t)
            pv = out_v.cpu().numpy()
            # Val loss on the same objective, so a declining rank can be read as
            # overfitting (val loss rises too) rather than the loss and the metric
            # disagreeing (val loss keeps falling while rank does not).
            vl = float(np.mean([float(clip_loss(out_v[i:i + batch], yv_t[i:i + batch]))
                                for i in range(0, len(out_v) - 7, batch)]))
        r = rank_pct(pv, yv, emb)
        if r > best:
            best, best_state = r, {k: v.detach().clone() for k, v in head.state_dict().items()}
        if ep % log_every == 0 or ep == 1:
            print(f'    epoch {ep:3d}  train loss {tot / len(Xf_t):.4f}  '
                  f'val loss {vl:.4f}  val rank {r:.4f}{"  *" if r == best else ""}')
    head.load_state_dict(best_state)
    return head, best


if RUN_CLIP_PROBE:
    t0 = time.time()
    print('Section A: ClipLoss on the frozen pooled features')
    headA, bestA = train_head(Xtr[~is_val], ytr[~is_val], Xtr[is_val], ytr[is_val],
                              A_EPOCHS, A_BATCH, A_LR)
    with torch.no_grad():
        predA = headA(torch.tensor(Xte, device=DEVICE)).cpu().numpy()
    print(f'  best val rank {bestA:.4f}   ({time.time() - t0:.0f}s)')
    resA = score(predA, yte, ste, emb, 'ClipLoss probe')
    print('  ridge baseline   rank 0.4998   top-5  2.48% (per-window)   2.50% (subject-agg)')
    torch.save({k: v.cpu() for k, v in headA.state_dict().items()}, WEIGHTS / 'head_clip.pt')
    save_result('clip_probe', resA)
    print(f'\n  saved -> {WEIGHTS / "head_clip.pt"}')
    print('  SECTION A DONE. Paste the two score lines above before running Section B.')
else:
    print('Section A skipped')

## Section B — LoRA on the attention projections

The encoder now trains, so the cached features are void and the windows are rebuilt from
the EDFs using the settings recorded in the bundle.

In [ ]:
import mne, pandas as pd
mne.set_log_level('ERROR')
STIM = {'train': 'stim_train', 'test': 'stim_test'}


def build_windows(rel, split):
    """Re-epoch one recording exactly as track1_02_features did."""
    sf = PP['sfreq']
    n_times = int(round(1.0 * sf))
    tmin, tmax = -0.2, -0.2 + 1.0 - 1.0 / sf
    raw = mne.io.read_raw_edf(RAW / DATASET / rel, preload=True)
    if PP['filt']:
        raw.filter(PP['filt'][0], PP['filt'][1], verbose=False)
    raw.resample(sf)
    if PP['scale_scope'] == 'recording':
        raw.apply_function(lambda x: (x - x.mean()) / (x.std() + 1e-8), channel_wise=True)

    tsv = RAW / DATASET / rel.replace('_eeg.edf', '_events.tsv')
    df = pd.read_csv(tsv, sep='\t')
    df = df[df.trial_type == STIM[split]].dropna(subset=['stim_file']).reset_index(drop=True)
    s = np.round(df.onset.values * sf).astype(int)
    lo, hi = int(round(tmin * sf)), int(round(tmax * sf))
    k = (s + lo >= 0) & (s + hi < raw.n_times)
    df, s = df[k].reset_index(drop=True), s[k]
    ev = np.column_stack([s, np.zeros_like(s), np.ones_like(s)])
    baseline = None if PP['baseline'] in ('None', 'none') else eval(PP['baseline'])
    epo = mne.Epochs(raw, ev, tmin=tmin, tmax=tmax, baseline=baseline,
                     preload=True, verbose=False)
    X = epo.get_data(copy=True).astype('float32')
    if PP['scale_scope'] == 'window':
        X = (X - X.mean(2, keepdims=True)) / (X.std(2, keepdims=True) + 1e-8)
    return np.clip(X, -PP['clamp'], PP['clamp']), df.stim_file.values[epo.selection]


if RUN_LORA or RUN_UNPOOLED:
    man = json.loads((PREP / 'prepare_manifest.json').read_text())
    key = {k: i for i, k in enumerate(t['keys'].tolist())}
    W = {'train': [[], [], []], 'test': [[], [], []]}
    t0 = time.time()
    for rel, meta_r in sorted(man['recordings'].items()):
        sp = meta_r['split']
        Xw, stim = build_windows(rel, sp)
        idx = np.array([key.get(s, -1) for s in stim])
        keep = idx >= 0
        W[sp][0].append(Xw[keep]); W[sp][1].append(idx[keep])
        W[sp][2].append(np.full(int(keep.sum()), meta_r['subject']))
    Wtr = [np.concatenate(a) for a in W['train']]
    Wte = [np.concatenate(a) for a in W['test']]
    CH = mne.io.read_raw_edf(RAW / DATASET / sorted(man['recordings'])[0],
                             preload=False).ch_names
    print(f'windows  train {Wtr[0].shape}  test {Wte[0].shape}   ({time.time() - t0:.0f}s)')
    print(f'memory   {(Wtr[0].nbytes + Wte[0].nbytes) / 1e9:.2f} GB float32')
else:
    print('Section B skipped')

In [ ]:
class LoRALinear(nn.Module):
    """Frozen base plus a rank-r update. B starts at zero, so training begins
    exactly at the pretrained function."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.zeros(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale


class RevePooledTrain(nn.Module):
    def __init__(self, chs_info, n_outputs, rank, alpha, pooling='mean'):
        super().__init__()
        self.pooling = pooling
        from braindecode.models import REVE
        self.net = REVE.from_pretrained(ENCODER, n_outputs=n_outputs,
                                        n_chans=len(chs_info), n_times=200,
                                        sfreq=200.0, chs_info=chs_info)
        self.net.final_layer = nn.Identity()
        for p in self.net.parameters():
            p.requires_grad_(False)
        targets = [n for n, m in self.net.named_modules()
                   if isinstance(m, nn.Linear) and n.split('.')[-1] in ('to_qkv', 'to_out')]
        for name in targets:
            parent = self.net.get_submodule(name.rsplit('.', 1)[0])
            setattr(parent, name.split('.')[-1],
                    LoRALinear(self.net.get_submodule(name), rank, alpha))
        self.n_lora = len(targets)
        self.head = nn.Linear(self.net.embed_dim, n_outputs)
        if pooling == 'attention':
            # A learned query over the channel-by-patch tokens. Mean pooling weights
            # every electrode equally, which on a visual task dilutes occipital signal
            # across 31 other channels; this learns the weighting instead. It is only
            # embed_dim parameters and stays independent of channel count.
            self.query = nn.Parameter(torch.randn(self.net.embed_dim) * 0.02)

    def forward(self, X):
        feats = self.net(X, return_features=True)['features']   # (B, chan, patch, emb)
        tok = feats.flatten(1, 2)                               # (B, chan*patch, emb)
        if self.pooling == 'attention':
            w = torch.softmax((tok @ self.query) / self.net.embed_dim ** 0.5, dim=1)
            pooled = (w.unsqueeze(-1) * tok).sum(1)
        else:
            pooled = tok.mean(1)
        return self.head(pooled)

In [ ]:
if RUN_LORA:
    CONFIGS = REG_SWEEP if CONFIG_SET == 'reg_sweep' else BASELINE

    ytr_w, subj_w = Wtr[1], Wtr[2]
    iv = np.array([y in val_imgs for y in ytr_w])
    Xf = torch.tensor(Wtr[0][~iv]); yf = torch.tensor(ytr_w[~iv].astype('int64'))
    Xv = torch.tensor(Wtr[0][iv]);  yv = ytr_w[iv]
    yv_t = torch.tensor(yv.astype('int64'))
    Xte_t = torch.tensor(Wte[0])
    print(f'fit {len(Xf):,} windows / val {len(Xv):,}')
    # A loss at these values means the model is not discriminating at all.
    print(f'chance loss: train {np.log(B_BATCH):.4f} (batch {B_BATCH}), '
          f'val {np.log(64):.4f} (batch 64)\n')

    def run_config(cfg):
        global NORM_PRED
        NORM_PRED = cfg['norm_pred']
        model = RevePooledTrain([{'ch_name': c} for c in CH], DIM, cfg['rank'],
                                2 * cfg['rank'],
                                pooling=cfg.get('pooling', 'mean')).to(DEVICE)
        # Warm start from Section A whether or not it ran this session. A random head
        # sends noise gradients into the encoder on the first steps, which is what the
        # reference's probe-then-LoRA order avoids.
        if RUN_CLIP_PROBE:
            model.head.load_state_dict(headA.state_dict())
        elif (WEIGHTS / 'head_clip.pt').exists():
            model.head.load_state_dict(torch.load(WEIGHTS / 'head_clip.pt',
                                                  map_location='cpu'))
        n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)

        opt = torch.optim.AdamW(
            [{'params': [p for n, p in model.named_parameters()
                         if p.requires_grad and not n.startswith('head')],
              'lr': cfg['lr']},
             {'params': model.head.parameters(), 'lr': cfg['head_lr']}],
            weight_decay=cfg['wd'])

        @torch.no_grad()
        def infer(X, bs=64):
            model.eval()
            return np.concatenate([model(X[i:i + bs].to(DEVICE)).float().cpu().numpy()
                                   for i in range(0, len(X), bs)])

        @torch.no_grad()
        def val_loss(X, bs=64):
            model.eval()
            out = [float(clip_loss(model(X[i:i + bs].to(DEVICE)),
                                   yv_t[i:i + bs].to(DEVICE)))
                   for i in range(0, len(X) - 7, bs)]
            return float(np.mean(out)) if out else float('nan')

        print(f"{cfg['name']}   {n_tr / 1e6:.2f}M trainable")
        best, best_state = -1, None
        for ep in range(1, cfg['epochs'] + 1):
            model.train()
            perm = torch.randperm(len(Xf))
            tot, t0 = 0.0, time.time()
            for i in range(0, len(perm) - 1, B_BATCH):
                idx = perm[i:i + B_BATCH]
                if len(idx) < 8:
                    continue
                opt.zero_grad()
                loss = clip_loss(model(Xf[idx].to(DEVICE)), yf[idx].to(DEVICE))
                loss.backward(); opt.step()
                tot += float(loss) * len(idx)
            r, vl = rank_pct(infer(Xv), yv, emb), val_loss(Xv)
            if r > best:
                best = r
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            print(f'  epoch {ep:2d}  train loss {tot / len(Xf):.4f}  val loss {vl:.4f}  '
                  f'val rank {r:.4f}{"  *" if r == best else ""}  {time.time() - t0:.0f}s')
        model.load_state_dict(best_state)
        if cfg.get('pooling') == 'attention':
            # Which electrodes did it learn to attend to? If the answer is occipital,
            # the mechanism is doing what the per-electrode result predicted.
            with torch.no_grad():
                tk = model.net(Xv[:256].to(DEVICE),
                               return_features=True)['features'].flatten(1, 2)
                wts = torch.softmax((tk @ model.query) / model.net.embed_dim ** 0.5,
                                    dim=1).mean(0).cpu().numpy()
            top = np.argsort(-wts)[:6]
            print('    attention mass on: '
                  + ', '.join(f'{CH[i % len(CH)]} {wts[i]:.3f}' for i in top)
                  + f'   (uniform would be {1 / len(wts):.3f})')
        pred = infer(Xte_t)
        del model
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        return best, best_state, pred

    runs = []
    for cfg in CONFIGS:
        bv, st, pred = run_config(cfg)
        res = score(pred, Wte[1], Wte[2], emb, cfg['name'])
        res['val_rank'] = bv
        res['config'] = cfg          # keep the LoRA rank out of res['rank'],
                                     # which is the rank_pct metric
        runs.append((cfg, res, st))
        print()

    print(f'{"config":<22}{"val rank":>10}{"test rank":>11}{"top-5":>9}')
    for cfg, res, _ in runs:
        print(f'{cfg["name"]:<22}{res["val_rank"]:10.4f}{res["rank"]:11.4f}'
              f'{res["top5"]:8.2f}%')
    prev = load_results()
    if 'clip_probe' in prev:
        print(f'{"ClipLoss probe":<22}{"":>10}{prev["clip_probe"]["rank"]:11.4f}'
              f'{prev["clip_probe"]["top5"]:8.2f}%')
    print(f'{"ridge":<22}{"":>10}{0.4998:11.4f}{2.48:8.2f}%')
    print(f'{"chance":<22}{"":>10}{0.5000:11.4f}{runs[0][1]["chance"]:8.2f}%')

    cfg, resB, best_state = max(runs, key=lambda t: t[1]['val_rank'])
    print(f'\nbest by val rank: {cfg["name"]}')
    save_result('lora', resB)
    torch.save(best_state, WEIGHTS / 'lora_model.pt')
    (WEIGHTS / 'lora_config.json').write_text(json.dumps(cfg, indent=2))
    print(f'  saved -> {WEIGHTS / "lora_model.pt"} '
          f'({(WEIGHTS / "lora_model.pt").stat().st_size / 1e6:.0f} MB)')
    print('  SECTION B DONE. Paste the epoch logs and the summary table.')


## Section D — does pooling cost us the signal?

Ridge from REVE's **unpooled** tokens: `32 channels x 512 = 16,384` features, against the
pooled 512 that every run so far has used. Closed form, so there is no training loop and
no hyperparameter to confound the comparison; lambda is swept and chosen on the same
image-wise validation split.

This is a measurement, not a submission path for the warm-up: a flatten head is fixed to
one channel count. It *is* shippable for the sealed phase, which the organisers confirmed
uses the same 32-channel montage.

The comparison that matters:

```
raw EEG, 3,200 samples   rank 0.5116   ~10.6 sigma
REVE pooled, 512         rank 0.4998   chance
REVE unpooled, 16,384    <- this
```

If unpooled lands near raw, pooling is where the signal goes and the encoder is fine. If
it lands near chance, the encoder is discarding the information before pooling ever
happens, and REVE is the wrong tool for a 300 ms response.


In [ ]:
if RUN_UNPOOLED:
    from braindecode.models import REVE
    t0 = time.time()
    enc = REVE.from_pretrained(ENCODER, n_outputs=DIM, n_chans=len(CH), n_times=200,
                               sfreq=200.0,
                               chs_info=[{'ch_name': c} for c in CH]).to(DEVICE).eval()
    enc.final_layer = torch.nn.Identity()

    @torch.no_grad()
    def tokens_of(X, bs=128):
        out = []
        for i in range(0, len(X), bs):
            f_ = enc(torch.from_numpy(X[i:i + bs]).to(DEVICE),
                     return_features=True)['features']
            out.append(f_.mean(dim=2).flatten(1).float().cpu().numpy())   # (b, 32*512)
        return np.concatenate(out)

    Utr, Ute = tokens_of(Wtr[0]), tokens_of(Wte[0])
    del enc
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'unpooled features {Utr.shape} / {Ute.shape}   ({time.time() - t0:.0f}s, '
          f'{(Utr.nbytes + Ute.nbytes) / 1e9:.2f} GB)')

    iv_u = np.array([y in val_imgs for y in Wtr[1]])
    Xa = torch.tensor(Utr[~iv_u], device=DEVICE)
    Yv = emb[Wtr[1][~iv_u]]
    mu_, sd_ = Xa.mean(0), Xa.std(0) + 1e-8
    Xa = (Xa - mu_) / sd_
    Ya = torch.tensor(Yv, device=DEVICE)
    ym_ = Ya.mean(0)
    Yc = Ya - ym_
    Xv_ = (torch.tensor(Utr[iv_u], device=DEVICE) - mu_) / sd_
    Xb_ = (torch.tensor(Ute, device=DEVICE) - mu_) / sd_

    # p (16,384) > n, so solve in dual form: K is n x n rather than p x p.
    K = Xa @ Xa.T
    scale = float(torch.diagonal(K).mean())
    Kv, Kb = Xv_ @ Xa.T, Xb_ @ Xa.T
    eye = torch.eye(K.shape[0], device=DEVICE)
    print(f'{"lambda":>10} {"val rank":>10}')
    best = (-1, None, None)
    for rel in np.logspace(-3, 2, 6):
        alpha = torch.linalg.solve(K + rel * scale * eye, Yc)
        pv = (Kv @ alpha + ym_).cpu().numpy()
        r = rank_pct(pv, Wtr[1][iv_u], emb)
        print(f'{rel:10.3g} {r:10.4f}')
        if r > best[0]:
            best = (r, rel, (Kb @ alpha + ym_).cpu().numpy())
    grid = np.logspace(-3, 2, 6)
    print(f'\nbest lambda {best[1]:.3g}, val rank {best[0]:.4f}')
    if best[1] in (grid[0], grid[-1]):
        print('  WARNING: lambda is at the edge of the grid, so the sweep is truncated.')
        print('  At the top edge the fit may also have collapsed toward a constant.')
    sp_ = float(np.linalg.norm(best[2] - best[2].mean(0), axis=1).mean()
                / (np.linalg.norm(best[2].mean(0)) + 1e-12))
    print(f'  prediction spread {sp_:.4f}'
          + ('   COLLAPSED' if sp_ < 1e-3 else ''))
    resD = score(best[2], Wte[1], Wte[2], emb, 'REVE unpooled')
    print(f'  {"REVE pooled":<16} rank 0.4998   top-5  2.48%')
    print(f'  {"raw EEG":<16} rank 0.5116   top-5  2.76%')
    save_result('unpooled_ridge', resD)
    del K, Kv, Kb, Xa, Xv_, Xb_, Utr, Ute
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print('\n  SECTION D DONE.')
else:
    print('Section D skipped')


## Section C — package the LoRA model

Builds the flat zip and then **loads it back the way the server will**, at a geometry it
was not trained on, before you spend a submission slot on it. A LoRA submission has to
rebuild the adapter modules before `load_state_dict`, so it needs its own `submission.py`;
the one in `track1_03_head` will not load these weights.

Runs on its own: it reads `lora_model.pt` and `results.json` from Drive rather than from
whatever is in memory.


In [ ]:
# Source of truth is solvers/track1_image_reve_lora.py in the repo.
SUBMISSION_PY = "\"\"\"Track 01 submission: REVE with LoRA on the attention projections, pooled head.\n\nEvery pretrained tensor is unchanged. LoRA freezes each attention projection `W` and\nlearns a rank-r correction `B\u00b7A` beside it, so the checkpoint's own weights are never\noverwritten; `B` trains from zero, meaning the model starts at exactly the pretrained\nfunction.\n\nAdapted modules are the 44 `transformer.layers.*.0.{to_qkv,to_out}` projections named\nin `neuralbench/models/reve.yaml`. The head pools tokens over channels and patches and\nmaps `embed_dim -> n_outputs`, which keeps it independent of channel count and window\nlength; REVE's own `final_layer` flattens and so is corpus-specific.\n\nInput is resampled to 200 Hz to match pretraining, z-scored per channel and clipped at\n15 SD, matching how the training windows were built.\n\"\"\"\n\nimport os\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import REVE\n\nMIN_TIMES = 200        # REVE's patch_size\nTRAIN_SFREQ = 200.0\nCLIP_SD = 15.0\nLORA_RANK = 16         # must match training\nLORA_ALPHA = 32\nLORA_TARGETS = (\"to_qkv\", \"to_out\")\n\n\ndef resample_fft(x, n_out):\n    \"\"\"Resample along time via the rfft spectrum. Pure torch: nothing is installed\n    at evaluation, and linear interpolation aliases when downsampling.\"\"\"\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass LoRALinear(nn.Module):\n    \"\"\"Frozen base plus a rank-r update. Structure must match training exactly, or\n    the state dict will not load.\"\"\"\n\n    def __init__(self, base, r, alpha):\n        super().__init__()\n        self.base = base\n        self.A = nn.Parameter(torch.zeros(r, base.in_features))\n        self.B = nn.Parameter(torch.zeros(base.out_features, r))\n        self.scale = alpha / r\n\n    def forward(self, x):\n        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale\n\n\nclass RevePooledLoRA(nn.Module):\n\n    def __init__(self, sfreq, n_outputs, chs_info, **kwargs):\n        super().__init__()\n        self.net = REVE(\n            n_outputs=n_outputs, sfreq=TRAIN_SFREQ, chs_info=chs_info, **kwargs\n        )\n        self.net.final_layer = nn.Identity()\n\n        names = [\n            n for n, m in self.net.named_modules()\n            if isinstance(m, nn.Linear) and n.split(\".\")[-1] in LORA_TARGETS\n        ]\n        for name in names:\n            parent = self.net.get_submodule(name.rsplit(\".\", 1)[0])\n            setattr(\n                parent,\n                name.split(\".\")[-1],\n                LoRALinear(self.net.get_submodule(name), LORA_RANK, LORA_ALPHA),\n            )\n        self.n_lora = len(names)\n        self.head = nn.Linear(self.net.embed_dim, n_outputs)\n        self.in_sfreq = float(sfreq)\n\n        # The position bank returns only the names it recognises and merely logs the\n        # shortfall, which later surfaces as a tensor-size mismatch in the forward.\n        # Resolve names here so the input width and the positions always agree.\n        bank = self.net._position_bank.mapping\n        folded = {k.casefold(): k for k in bank}\n        keep, resolved = [], []\n        for i, ch in enumerate(c[\"ch_name\"] for c in chs_info):\n            name = ch if ch in bank else folded.get(ch.casefold())\n            if name is not None:\n                keep.append(i)\n                resolved.append(name)\n        if not keep:\n            raise RuntimeError(\n                \"no channel name resolved against REVE's position bank: \"\n                f\"{[c['ch_name'] for c in chs_info][:8]}\"\n            )\n        self.register_buffer(\"keep_idx\", torch.tensor(keep), persistent=False)\n        self.register_buffer(\"pos\", self.net.get_positions(resolved), persistent=False)\n\n    def forward(self, X):\n        X = X.index_select(1, self.keep_idx)\n        n_out = max(int(round(X.shape[-1] * TRAIN_SFREQ / self.in_sfreq)), MIN_TIMES)\n        X = resample_fft(X, n_out)\n        mu = X.mean(dim=-1, keepdim=True)\n        sd = X.std(dim=-1, keepdim=True)\n        X = torch.clamp((X - mu) / (sd + 1e-8), -CLIP_SD, CLIP_SD)\n        pos = self.pos.unsqueeze(0).expand(X.shape[0], -1, -1)\n        feats = self.net(X, pos=pos, return_features=True)[\"features\"]\n        return self.head(feats.mean(dim=(1, 2)))\n\n    @torch.inference_mode()\n    def predict(self, X):\n        self.eval()\n        return self(X)\n\n\nclass Solver(CompetSolver):\n\n    name = \"REVE-base-lora-qkvo-clip\"\n\n    def load_model(self, meta):\n        os.environ[\"REVE_POSITIONS_PATH\"] = str(meta[\"submission_dir\"])\n\n        model = RevePooledLoRA(\n            sfreq=meta[\"sfreq\"],\n            n_outputs=meta[\"n_outputs\"],\n            chs_info=meta[\"chs_info\"],\n            n_chans=meta[\"n_chans\"],\n            n_times=MIN_TIMES,   # forward is dynamic; this only sized the dropped head\n        )\n        state = torch.load(\n            meta[\"submission_dir\"] / \"weights.pt\",\n            map_location=meta[\"device\"],\n            weights_only=True,\n        )\n        missing, unexpected = model.load_state_dict(state, strict=False)\n        if unexpected:\n            raise RuntimeError(f\"unexpected keys in weights.pt: {unexpected[:5]}\")\n        if missing:\n            raise RuntimeError(f\"weights missing: {missing[:5]}\")\n        return model.to(meta[\"device\"]).eval()\n"
print(f'submission.py {len(SUBMISSION_PY.splitlines())} lines')


In [ ]:
if MAKE_ZIP:
    import importlib.util, sys, types

    res = load_results()
    best = res.get('lora') or res.get('clip_probe') or {}
    if not best:
        raise RuntimeError('no results.json on Drive; run Section A or B first')
    print(f'best per-window top-5 {best["top5"]:.2f}% vs chance {best["chance"]:.2f}%')
    if best['top5'] <= best['chance'] and not globals().get('FORCE_ZIP'):
        print('\nAt or below chance, so a submission would only reproduce the floor')
        print('already on the board. Set FORCE_ZIP = True to override.')
    else:
        out = Path(ZIP_DIR)
        if out.exists():
            shutil.rmtree(out)
        out.mkdir(parents=True)

        state = torch.load(WEIGHTS / 'lora_model.pt', map_location='cpu')
        torch.save(state, out / 'weights.pt')

        # The solver rebuilds the adapters before load_state_dict, so its rank must
        # match whatever Section B settled on. Write the chosen values in rather than
        # trusting the file's defaults.
        cfg_path = WEIGHTS / 'lora_config.json'
        src = SUBMISSION_PY
        if cfg_path.exists():
            cfg_used = json.loads(cfg_path.read_text())
            src = src.replace('LORA_RANK = 16', f'LORA_RANK = {cfg_used["rank"]}')
            src = src.replace('LORA_ALPHA = 32', f'LORA_ALPHA = {2 * cfg_used["rank"]}')
            print(f'solver pinned to rank {cfg_used["rank"]} '
                  f'(alpha {2 * cfg_used["rank"]}) from {cfg_path.name}')
        else:
            print('no lora_config.json; solver keeps its default rank 16')
        (out / 'submission.py').write_text(src)
        cand = list(Path('/root/mne_data').glob('*reve*positions*.json')) + \
               list(Path.home().glob('mne_data/*reve*positions*.json'))
        if cand:
            shutil.copy(cand[0], out / 'reve_positions.json')
        else:
            import huggingface_hub
            shutil.copy(huggingface_hub.hf_hub_download('brain-bzh/reve-positions',
                                                        'positions.json'),
                        out / 'reve_positions.json')

        # Load it the way the server does, with a stand-in for the harness base class.
        stub = types.ModuleType('benchmark_utils')
        sub = types.ModuleType('benchmark_utils.base_solver')
        sub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
        stub.base_solver = sub
        sys.modules.setdefault('benchmark_utils', stub)
        sys.modules.setdefault('benchmark_utils.base_solver', sub)
        spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)

        checks = []
        for label, chs, sf, nt in (('trained geometry', CH, 200.0, 200),
                                   ('native 256 Hz', CH, 256.0, 256),
                                   ('half the channels', CH[::2], 200.0, 200)):
            try:
                m = mod.Solver().load_model(
                    {'sfreq': sf, 'n_outputs': DIM, 'n_chans': len(chs),
                     'chs_info': [{'ch_name': c} for c in chs],
                     'submission_dir': out, 'device': 'cpu'})
                o = m.predict(torch.randn(2, len(chs), nt))
                checks.append((label, tuple(o.shape) == (2, DIM) and bool(torch.isfinite(o).all()),
                               f'{tuple(o.shape)}'))
            except Exception as e:
                checks.append((label, False, f'{type(e).__name__}: {str(e)[:70]}'))

        zpath = shutil.make_archive(str(out.parent / SUBMISSION_NAME), 'zip', out)
        checks.append(('flat zip', True, f'{Path(zpath).stat().st_size / 1e6:.0f} MB'))
        report('submission:', checks)
        print(f'\n{zpath}')
        print('Download from the Colab file browser, then upload to Codabench.')
else:
    print('Section C skipped')
